In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [3]:
from datasets import load_dataset

dataset = load_dataset("blinoff/kinopoisk")

1. Что хранится в blinoff/kinopoisk:

    Датасет содержит 36 591 отзыв о фильмах с сайта Кинопоиск, собранный из списков "Топ-250" и "Худшие 100" (Bottom-100).

    Период: с июля 2004 года по ноябрь 2012 года.

    Формат хранения на Hugging Face: Apache Arrow (Parquet под капотом), что обеспечивает быструю загрузку и эффективную работу с памятью через библиотеку datasets.
    
    Основная задача: Многоклассовая классификация тональности текста (3 класса).

In [4]:
from collections import Counter

# Загружаем данные
dataset = load_dataset("blinoff/kinopoisk", split="train[:5%]")

# 1. Анализ пропусков (нативными средствами)
# Считаем количество None или пустых строк в колонке 'title'
missing_titles = sum(1 for x in dataset['title'] if x is None or x.strip() == "")
print(f"Пропусков в 'title': {missing_titles} из {len(dataset)}")

# 2. Анализ дисбаланса классов
# Используем Counter для быстрого подсчета
grade_counts = Counter(dataset['grade3'])
total = len(dataset)
print("\nРаспределение классов:")
for grade, count in grade_counts.items():
    print(f"  {grade}: {count} ({count/total:.2%})")

# 3. Статистика длин текстов (без создания новой колонки в памяти)
lengths = [len(text) for text in dataset['content'] if text is not None]
print(f"\nДлина текста 'content':")
print(f"  Мин: {min(lengths)}")
print(f"  Макс: {max(lengths)}")
print(f"  Среднее: {sum(lengths)/len(lengths):.0f}")

# 4. Поиск дубликатов (пример для Подсистемы 1)
# Считаем уникальные тексты
unique_contents = len(set(dataset['content']))
print(f"\nУникальных отзывов: {unique_contents} из {len(dataset)}")
print(f"Дубликатов: {len(dataset) - unique_contents}")

Пропусков в 'title': 415 из 1830

Распределение классов:
  Good: 1601 (87.49%)
  Neutral: 160 (8.74%)
  Bad: 69 (3.77%)

Длина текста 'content':
  Мин: 62
  Макс: 11830
  Среднее: 1822

Уникальных отзывов: 1829 из 1830
Дубликатов: 1


### EDA

In [5]:
print("\n--- ЭТАП 1: Анализ пропусков (Надежный вариант) ---")

# Инициализируем словарь для подсчета
total_missing = {col: 0 for col in dataset.column_names}

# Итерируемся по датасету батчами (это быстро и экономит память, как и map)
for batch in dataset.iter(batch_size=1000):
    for col in dataset.column_names:
        for val in batch[col]:
            # Проверяем на None, пустую строку или строку из пробелов
            if val is None or (isinstance(val, str) and val.strip() == ""):
                total_missing[col] += 1

print("Количество и процент пропусков:")
for col, count in total_missing.items():
    pct = (count / len(dataset)) * 100
    print(f"  {col:12}: {count:5} ({pct:5.2f}%)")


--- ЭТАП 1: Анализ пропусков (Надежный вариант) ---
Количество и процент пропусков:
  part        :     0 ( 0.00%)
  movie_name  :     0 ( 0.00%)
  review_id   :     0 ( 0.00%)
  author      :    48 ( 2.62%)
  date        :     0 ( 0.00%)
  title       :   415 (22.68%)
  grade3      :     0 ( 0.00%)
  grade10     :     0 ( 0.00%)
  content     :     0 ( 0.00%)


#### Блок 2: Анализ дисбаланса классов (Target Variable)

In [6]:
print("\n--- ЭТАП 2: Анализ дисбаланса классов (Оптимизировано) ---")
from collections import Counter

total_rows = len(dataset)

# Обращаемся к колонке напрямую и считаем уникальные значения через Counter
# Это работает в разы быстрее, чем dataset.filter()
grade_counts = Counter(dataset["grade3"])

print(f"Всего записей: {total_rows}")
print("Распределение по классам:")

for cls, count in grade_counts.items():
    pct = (count / total_rows) * 100
    print(f"  {cls:8}: {count:5} ({pct:5.2f}%)")


--- ЭТАП 2: Анализ дисбаланса классов (Оптимизировано) ---
Всего записей: 1830
Распределение по классам:
  Good    :  1601 (87.49%)
  Neutral :   160 ( 8.74%)
  Bad     :    69 ( 3.77%)


#### Блок 3: Статистика длин текстов (Feature Engineering)

In [7]:
print("\n--- ЭТАП 3: Статистика длин текстов ---")

# Добавляем колонку с длиной контента
# num_proc=-1 использует все доступные ядра CPU для ускорения
dataset = dataset.map(
    lambda x: {"content_len": len(x["content"]) if x["content"] else 0},
    num_proc=1 # Используйте -1 для больших датасетов, но для 10% достаточно и 1
)

# Извлекаем длины в список для статистики (это безопасно, так как это просто числа)
lengths = dataset["content_len"]

print(f"Статистика длины отзывов (в символах):")
print(f"  Минимум : {min(lengths)}")
print(f"  Максимум: {max(lengths)}")
print(f"  Среднее : {sum(lengths) / len(lengths):.0f}")

# Считаем медиану и перцентили без сортировки всего списка (используем квантили)
sorted_lengths = sorted(lengths)
n = len(sorted_lengths)
print(f"  Медиана : {sorted_lengths[n // 2]}")
print(f"  95-й перцентиль: {sorted_lengths[int(n * 0.95)]}")


--- ЭТАП 3: Статистика длин текстов ---
Статистика длины отзывов (в символах):
  Минимум : 62
  Максимум: 11830
  Среднее : 1822
  Медиана : 1551
  95-й перцентиль: 4016


#### Блок 4: Анализ временного дрейфа (Temporal Drift)

In [8]:
import datetime
print("\n--- ЭТАП 4: Анализ временного дрейфа ---")

# Функция для извлечения года из даты (формат может быть "YYYY-MM-DD" или timestamp)
def extract_year(date_val):
    if date_val is None:
        return None
    if isinstance(date_val, datetime.datetime):
        return date_val.year
    if isinstance(date_val, str):
        try:
            return int(date_val.split("-")[0])
        except:
            return None
    return None

# Добавляем колонку с годом
dataset = dataset.map(lambda x: {"year": extract_year(x["date"])})

# Фильтруем записи, где год успешно извлечен
dataset_with_years = dataset.filter(lambda x: x["year"] is not None)

# Смотрим, как менялась доля негативных отзывов по годам
years = sorted(dataset_with_years.unique("year"))
print("Динамика доли негативных отзывов (Bad) по годам:")

for year in years:
    year_data = dataset_with_years.filter(lambda x: x["year"] == year)
    bad_count = len(year_data.filter(lambda x: x["grade3"] == "Bad"))
    total_year = len(year_data)
    bad_pct = (bad_count / total_year * 100) if total_year > 0 else 0
    print(f"  {year}: {bad_pct:5.2f}% негативных (всего отзывов: {total_year})")


--- ЭТАП 4: Анализ временного дрейфа ---
Динамика доли негативных отзывов (Bad) по годам:
  2005:  0.00% негативных (всего отзывов: 18)
  2006:  9.52% негативных (всего отзывов: 63)
  2007:  2.22% негативных (всего отзывов: 90)
  2008:  5.66% негативных (всего отзывов: 212)
  2009:  4.24% негативных (всего отзывов: 236)
  2010:  2.67% негативных (всего отзывов: 674)
  2011:  3.91% негативных (всего отзывов: 435)
  2012:  3.92% негативных (всего отзывов: 102)


#### Блок 5: Выявление дубликатов (Data Leakage & Noise)

In [9]:
print("\n--- ЭТАП 5: Поиск дубликатов контента ---")

# Извлекаем все тексты в set для поиска уникальных
# Это требует памяти, равной объему текстов, но для 10% это нормально
unique_contents = set(dataset["content"])

total = len(dataset)
unique_count = len(unique_contents)
duplicates_count = total - unique_count

print(f"Всего отзывов: {total}")
print(f"Уникальных текстов: {unique_count}")
print(f"Дубликатов: {duplicates_count} ({duplicates_count/total*100:.2f}%)")

# Дополнительно: проверим, есть ли отзывы с одинаковым текстом, но РАЗНЫМИ оценками (шумные лейблы)
print("\nПоиск противоречий (одинаковый текст, разные оценки)...")
text_to_grades = {}
for text, grade in zip(dataset["content"], dataset["grade3"]):
    if text not in text_to_grades:
        text_to_grades[text] = set()
    text_to_grades[text].add(grade)

conflicts = sum(1 for grades in text_to_grades.values() if len(grades) > 1)
print(f"Найдено {conflicts} текстов, которым присвоены разные оценки (шумные данные).")


--- ЭТАП 5: Поиск дубликатов контента ---
Всего отзывов: 1830
Уникальных текстов: 1829
Дубликатов: 1 (0.05%)

Поиск противоречий (одинаковый текст, разные оценки)...


Найдено 0 текстов, которым присвоены разные оценки (шумные данные).


#### Блок 6: Корреляция "Длина текста vs Оценка"

In [10]:
print("\n--- ЭТАП 6: Зависимость длины текста от оценки ---")

classes = dataset.unique("grade3")

for cls in classes:
    # Фильтруем по классу и извлекаем длины
    cls_lengths = dataset.filter(lambda x: x["grade3"] == cls)["content_len"]
    
    if cls_lengths:
        avg_len = sum(cls_lengths) / len(cls_lengths)
        sorted_cls_lens = sorted(cls_lengths)
        median_len = sorted_cls_lens[len(sorted_cls_lens) // 2]
        
        print(f"Класс '{cls}':")
        print(f"  Средняя длина: {avg_len:.0f} символов")
        print(f"  Медианная длина: {median_len} символов")


--- ЭТАП 6: Зависимость длины текста от оценки ---
Класс 'Good':
  Средняя длина: 1829 символов
  Медианная длина: 1561 символов
Класс 'Neutral':
  Средняя длина: 1640 символов
  Медианная длина: 1478 символов
Класс 'Bad':
  Средняя длина: 2083 символов
  Медианная длина: 1614 символов


### Подготовка

#### Блок 1: Фильтрация и очистка данных

In [11]:
import pandas as pd
from datasets import load_dataset

print("--- ЭТАП 1: Фильтрация и очистка данных ---")

# Загружаем полный датасет (или используем уже загруженный)
dataset = load_dataset("blinoff/kinopoisk", split="train")
print(f"Исходный размер: {len(dataset)} записей")

# Удаляем дубликаты по content
# datasets не имеет прямого метода remove_duplicates, поэтому используем pandas для этой операции
df = dataset.to_pandas()
initial_count = len(df)
df = df.drop_duplicates(subset=['content'])
print(f"Удалено дубликатов: {initial_count - len(df)}")
print(f"Размер после удаления дубликатов: {len(df)} записей")

# Обработка пропусков
df['title'] = df['title'].fillna('Без заголовка')
df['author'] = df['author'].fillna('Аноним')

# Проверяем пропуски в целевой переменной
df = df.dropna(subset=['grade3', 'content'])
print(f"Размер после удаления пропусков в grade3/content: {len(df)} записей")

# Возвращаем в формат datasets для дальнейшей работы
dataset = dataset.from_pandas(df)
print(f"Финальный размер датасета: {len(dataset)} записей\n")

--- ЭТАП 1: Фильтрация и очистка данных ---
Исходный размер: 36591 записей
Удалено дубликатов: 12
Размер после удаления дубликатов: 36579 записей
Размер после удаления пропусков в grade3/content: 36579 записей
Финальный размер датасета: 36579 записей



#### Блок 2: Feature Engineering (создание мета-признаков)

In [12]:
import re

print("--- ЭТАП 2: Feature Engineering ---")

def create_meta_features(batch):
    """Создание мета-признаков для каждого отзыва"""
    meta_features = {
        'content_len': [],
        'has_title': [],
        'has_numbers': []
    }
    
    for content, title in zip(batch['content'], batch['title']):
        # Длина текста
        meta_features['content_len'].append(len(content) if content else 0)
        
        # Наличие заголовка (не стандартного)
        has_title = 1 if (title and title != 'Без заголовка' and len(title.strip()) > 0) else 0
        meta_features['has_title'].append(has_title)
        
        # Наличие цифр в тексте
        has_numbers = 1 if re.search(r'\d', content) else 0
        meta_features['has_numbers'].append(has_numbers)
    
    return meta_features

# Применяем функцию ко всему датасету
dataset = dataset.map(
    create_meta_features,
    batched=True,
    batch_size=1000
)

print("Добавлены признаки: content_len, has_title, has_numbers")
print(f"Пример: {dataset[0]['content_len']} символов, заголовок: {dataset[0]['has_title']}, цифры: {dataset[0]['has_numbers']}\n")

--- ЭТАП 2: Feature Engineering ---


Map: 100%|██████████| 36579/36579 [00:01<00:00, 30100.88 examples/s]

Добавлены признаки: content_len, has_title, has_numbers
Пример: 2337 символов, заголовок: 1, цифры: 1



#### Блок 3: Текстовая нормализация

In [13]:
%pip install pymorphy3 nltk

Note: you may need to restart the kernel to use updated packages.


In [14]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Nightkin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
import re
from datasets import load_dataset

print("--- ЭТАП 3: Текстовая нормализация (самодостаточный) ---")

# ==========================================================================
# Переключатель: для LR + TF-IDF разница с лемматизацией часто невелика.
# Если нужно быстро - поставьте False, пайплайн продолжит работать.
# ==========================================================================
USE_LEMMA = True

# Стоп-слова захардкожены намеренно: не зависим от nltk.download()
RU_STOP = {
    'и','в','во','не','что','он','на','я','с','со','как','а','то','все','всё',
    'она','они','так','его','но','да','ты','к','у','же','вы','за','бы','по',
    'только','ее','её','мне','было','вот','от','меня','еще','ещё','нет','о','об',
    'из','ему','теперь','меня','нас','вам','вас','для','или','ни','до','его','чтобы',
    'этот','эта','это','эти','того','тоже','если','когда','чем','где','чтоб','какой',
    'при','после','перед','будет','был','была','были','есть','быть','уже','про',
    'себя','свой','свое','свою','который','которая','которое','которые','сам','сама',
    'также','тогда','почему','какие','много','мало','очень','просто','весь','вся',
    'мое','моей','мои','мой','наш','ваш','их','его','ей','нем','ней','ним','нею'
}


def preprocess(batch):
    """
    ПОЛНОСТЬЮ самодостаточная функция: все зависимости внутри,
    поэтому сериализация в datasets не может сломать имена.
    Кэш и лемматизатор хранятся как атрибуты самой функции.
    """
    # --- ленивая инициализация (выполнится один раз на первый батч) ---
    if not hasattr(preprocess, "_cache"):
        preprocess._cache = {}
        preprocess._re_url  = re.compile(r"(https?://|www\.)\S+")
        preprocess._re_html = re.compile(r"<[^>]+>")
        preprocess._re_num  = re.compile(r"\d+")
        # оставляем только буквы и пробелы (цифры убираем - для них есть мета-признак has_numbers)
        preprocess._re_bad  = re.compile(r"[^a-zа-яё\s]+", re.IGNORECASE)
        preprocess._re_ws   = re.compile(r"\s+")
        preprocess._morph   = None
        if USE_LEMMA:
            try:
                from pymorphy3 import MorphAnalyzer
                preprocess._morph = MorphAnalyzer(lang="ru")
                print("[ok] pymorphy3: лемматизация включена")
            except Exception as e:
                print(f"[warn] pymorphy3 недоступен ({e}) -> лемматизация ОТКЛЮЧЕНА")

    morph, cache = preprocess._morph, preprocess._cache
    out = []

    for raw in batch["content"]:
        # ---------- 1. грубая очистка ----------
        t = (raw or "").lower()
        t = preprocess._re_url.sub(" ", t)
        t = preprocess._re_html.sub(" ", t)
        t = preprocess._re_num.sub(" ", t)
        t = preprocess._re_bad.sub(" ", t)
        t = preprocess._re_ws.sub(" ", t).strip()

        # ---------- 2. токенизация ----------
        tokens = t.split()

        # ---------- 3. лемматизация + фильтрация (с кэшем!) ----------
        if morph is not None:
            lemms = []
            for w in tokens:
                lemma = cache.get(w)
                if lemma is None:
                    lemma = morph.parse(w)[0].normal_form
                    cache[w] = lemma
                if lemma in RU_STOP or len(lemma) < 3:
                    continue
                lemms.append(lemma)
            out.append(" ".join(lemms))
        else:
            out.append(" ".join(
                w for w in tokens if w not in RU_STOP and len(w) > 2
            ))

    return {"content_clean": out}


# ВАЖНО: num_proc НЕ указываем - любой пул процессов только ускоряет появление таких багов
dataset = dataset.map(preprocess, batched=True, batch_size=512)

# --------------------------------------------------------------------------
print("\nРезультат:")
print(f"  Размер кэша уникальных токенов: {len(preprocess._cache):,}")
sample = dataset[0]
print(f"\n  ОРИГИНАЛ : {sample['content'][:180]}...")
print(f"  ЧИСТЫЙ   : {sample['content_clean'][:180]}...")
avg_len = sum(len(x) for x in dataset['content_clean']) / len(dataset)
print(f"\n  Средняя длина чистого текста: {avg_len:.0f} симв.")
print("\n✓ Колонка 'content_clean' готова - её подаём в TfidfVectorizer (Блок 4)")

--- ЭТАП 3: Текстовая нормализация (самодостаточный) ---


Map:   0%|          | 0/36579 [00:00<?, ? examples/s]

[ok] pymorphy3: лемматизация включена


Map: 100%|██████████| 36579/36579 [00:24<00:00, 1471.45 examples/s]



Результат:
  Размер кэша уникальных токенов: 270,359

  ОРИГИНАЛ : 
"Блеф» — одна из моих самых любимых комедий.

Этот фильм я наверно смотрел раз сто, нет я конечно блефую, я видел его куда больше. Не могу не выразить своё восхищение главными дей...
  ЧИСТЫЙ   : блеф один самый любимый комедия фильм наверно смотреть раз сто конечно блефовать видеть куда большой мочь выразить восхищение главный действовать лицо фильм начать адриано челентан...

  Средняя длина чистого текста: 1458 симв.

✓ Колонка 'content_clean' готова - её подаём в TfidfVectorizer (Блок 4)


#### Блок 4: Векторизация текста (TF-IDF)

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
import numpy as np

print("--- ЭТАП 4: Векторизация текста ---")

# Извлекаем данные из datasets в numpy массивы для sklearn
X_text = np.array(dataset['content_clean'])
X_meta = np.column_stack([
    dataset['content_len'],
    dataset['has_title'],
    dataset['has_numbers']
])
y = np.array(dataset['grade3'])

print(f"Размер текстовых данных: {X_text.shape}")
print(f"Размер мета-признаков: {X_meta.shape}")

# Создание TF-IDF векторизатора
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),      # Униграммы и биграммы
    max_features=50000,      # Максимум 50k признаков
    min_df=3,                # Минимальная частота слова: 3
    max_df=0.9,              # Максимальная частота: 90% документов
    sublinear_tf=True        # Применение логарифмического масштабирования TF
)

# Векторизация текста
X_tfidf = tfidf.fit_transform(X_text)
print(f"Размер TF-IDF матрицы: {X_tfidf.shape}")
print(f"Размер словаря: {len(tfidf.vocabulary_)}")

# Масштабирование мета-признаков
scaler = StandardScaler()
X_meta_scaled = scaler.fit_transform(X_meta)

print("Мета-признаки масштабированы\n")

--- ЭТАП 4: Векторизация текста ---
Размер текстовых данных: (36579,)
Размер мета-признаков: (36579, 3)
Размер TF-IDF матрицы: (36579, 50000)
Размер словаря: 50000
Мета-признаки масштабированы



#### Блок 5: Разбиение данных и подготовка к обучению

In [17]:
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack
from sklearn.preprocessing import StandardScaler

print("--- ЭТАП 5: Разбиение данных и подготовка к обучению ---")

# ==========================================================================
# 1. Извлекаем данные из datasets
# ==========================================================================
X_text = np.array(dataset['content_clean'])
X_meta_original = np.column_stack([  # ← ОРИГИНАЛЬНЫЕ (немасштабированные)
    dataset['content_len'],
    dataset['has_title'],
    dataset['has_numbers']
])
y = np.array(dataset['grade3'])

print(f"Размер текстовых данных: {X_text.shape}")
print(f"Размер мета-признаков (оригинал): {X_meta_original.shape}")

# ==========================================================================
# 2. Стратифицированное разбиение на train/test (80/20)
# ==========================================================================
X_text_train, X_text_test, X_meta_train_orig, X_meta_test_orig, y_train, y_test = train_test_split(
    X_text, X_meta_original, y,  # ← Используем оригинальные мета-признаки
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nРазбиение на train/test выполнено со стратификацией")
print(f"Train set: {X_text_train.shape[0]} записей")
print(f"Test set: {X_text_test.shape[0]} записей")

# ==========================================================================
# 3. Векторизация текста (TF-IDF)
# ==========================================================================
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    min_df=3,
    max_df=0.9,
    sublinear_tf=True
)

X_text_train_tfidf = tfidf.fit_transform(X_text_train)
X_text_test_tfidf = tfidf.transform(X_text_test)

print(f"\nРазмер TF-IDF матрицы (train): {X_text_train_tfidf.shape}")
print(f"Размер TF-IDF матрицы (test): {X_text_test_tfidf.shape}")

# ==========================================================================
# 4. Масштабирование мета-признаков (ТОЛЬКО для обучения)
# ==========================================================================
scaler = StandardScaler()
X_meta_train_scaled = scaler.fit_transform(X_meta_train_orig)
X_meta_test_scaled = scaler.transform(X_meta_test_orig)

print("\nМета-признаки масштабированы для обучения")

# ==========================================================================
# 5. Объединение признаков
# ==========================================================================
X_train = hstack([X_text_train_tfidf, X_meta_train_scaled])
X_test = hstack([X_text_test_tfidf, X_meta_test_scaled])

print(f"\nФинальный размер признаков:")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")

# ==========================================================================
# 6. Проверка распределения классов
# ==========================================================================
from collections import Counter

train_class_dist = Counter(y_train)
test_class_dist = Counter(y_test)

print(f"\nРаспределение классов в train:")
for cls, count in sorted(train_class_dist.items()):
    print(f"  {cls}: {count} ({count/len(y_train)*100:.2f}%)")

print(f"\nРаспределение классов в test:")
for cls, count in sorted(test_class_dist.items()):
    print(f"  {cls}: {count} ({count/len(y_test)*100:.2f}%)")

# ==========================================================================
# 7. Сохраняем оригинальные мета-признаки для анализа
# ==========================================================================
print("\n" + "=" * 70)
print("ВАЖНО: Сохранены оригинальные мета-признаки для анализа")
print("=" * 70)
print(f"X_meta_train_orig: {X_meta_train_orig.shape} (немасштабированные)")
print(f"X_meta_test_orig: {X_meta_test_orig.shape} (немасштабированные)")
print("\nИспользуйте X_meta_test_orig для анализа (длина текста, заголовки, цифры)")
print("Используйте X_test для обучения модели (масштабированные)")

print("\n✓ Данные готовы для обучения логистической регрессии!")

--- ЭТАП 5: Разбиение данных и подготовка к обучению ---
Размер текстовых данных: (36579,)
Размер мета-признаков (оригинал): (36579, 3)

Разбиение на train/test выполнено со стратификацией
Train set: 29263 записей
Test set: 7316 записей

Размер TF-IDF матрицы (train): (29263, 50000)
Размер TF-IDF матрицы (test): (7316, 50000)

Мета-признаки масштабированы для обучения

Финальный размер признаков:
X_train: (29263, 50003)
X_test: (7316, 50003)

Распределение классов в train:
  Bad: 3801 (12.99%)
  Good: 21802 (74.50%)
  Neutral: 3660 (12.51%)

Распределение классов в test:
  Bad: 950 (12.99%)
  Good: 5451 (74.51%)
  Neutral: 915 (12.51%)

ВАЖНО: Сохранены оригинальные мета-признаки для анализа
X_meta_train_orig: (29263, 3) (немасштабированные)
X_meta_test_orig: (7316, 3) (немасштабированные)

Используйте X_meta_test_orig для анализа (длина текста, заголовки, цифры)
Используйте X_test для обучения модели (масштабированные)

✓ Данные готовы для обучения логистической регрессии!


### Логистическая регрессия

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score
import time

print("=" * 70)
print("БЛОК 6: Базовое обучение логистической регрессии")
print("=" * 70)

# ==========================================================================
# 1. Инициализация модели
# ==========================================================================
print("\n1. Инициализация модели...")

# ИСПРАВЛЕНО: solver='saga' вместо 'liblinear'
# saga поддерживает:
#   - Многоклассовую классификацию (multinomial) из коробки
#   - Разреженные матрицы (TF-IDF)
#   - Все типы регуляризации (l1, l2, elasticnet)
#   - class_weight='balanced'

model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    solver='saga',           # ← ИСПРАВЛЕНО
    penalty='l2',
    C=1.0,
    random_state=42,
    verbose=0,
)

print(f"   Модель: LogisticRegression")
print(f"   Параметры:")
print(f"      - solver='saga' (поддерживает multiclass)")
print(f"      - class_weight='balanced' (борьба с дисбалансом)")
print(f"      - penalty='l2', C=1.0")
print(f"      - multi_class='multinomial'")

# ==========================================================================
# 2. Обучение модели
# ==========================================================================
print("\n2. Обучение модели...")
start_time = time.time()

model.fit(X_train, y_train)

train_time = time.time() - start_time
print(f"   ✓ Обучение завершено за {train_time:.2f} сек.")

# ==========================================================================
# 3. Предсказание на тестовой выборке
# ==========================================================================
print("\n3. Предсказание на тестовой выборке...")
start_time = time.time()

y_pred = model.predict(X_test)

pred_time = time.time() - start_time
print(f"   ✓ Предсказание завершено за {pred_time:.2f} сек.")
print(f"   Размер предсказаний: {len(y_pred)}")

# ==========================================================================
# 4. Вычисление метрик качества
# ==========================================================================
print("\n4. Вычисление метрик качества...")

# Accuracy (для справки)
accuracy = accuracy_score(y_test, y_pred)

# F1-score с разными усреднениями
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')
f1_micro = f1_score(y_test, y_pred, average='micro')

print("\n" + "=" * 70)
print("РЕЗУЛЬТАТЫ БАЗОВОГО ОБУЧЕНИЯ")
print("=" * 70)

print(f"\n📊 Глобальные метрики:")
print(f"   Accuracy (точность)     : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   Macro F1-score          : {f1_macro:.4f} ← ГЛАВНАЯ МЕТРИКА")
print(f"   Weighted F1-score       : {f1_weighted:.4f}")
print(f"   Micro F1-score          : {f1_micro:.4f}")

# ==========================================================================
# 5. Детальный classification_report
# ==========================================================================
print("\n" + "=" * 70)
print("ДЕТАЛЬНЫЙ ОТЧЁТ ПО КЛАССАМ")
print("=" * 70)

report = classification_report(
    y_test, 
    y_pred, 
    target_names=['Bad', 'Good', 'Neutral'],
    digits=4
)

print("\n" + report)

# ==========================================================================
# 6. Анализ распределения предсказаний
# ==========================================================================
print("=" * 70)
print("РАСПРЕДЕЛЕНИЕ ПРЕДСКАЗАНИЙ")
print("=" * 70)

from collections import Counter

pred_dist = Counter(y_pred)
true_dist = Counter(y_test)

print(f"\nИстинное распределение (y_test):")
for cls in sorted(true_dist.keys()):
    count = true_dist[cls]
    pct = count / len(y_test) * 100
    print(f"   {cls:10}: {count:5} ({pct:5.2f}%)")

print(f"\nПредсказанное распределение (y_pred):")
for cls in sorted(pred_dist.keys()):
    count = pred_dist[cls]
    pct = count / len(y_pred) * 100
    print(f"   {cls:10}: {count:5} ({pct:5.2f}%)")

# ==========================================================================
# 7. Интерпретация результатов
# ==========================================================================
print("\n" + "=" * 70)
print("ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТОВ")
print("=" * 70)

print(f"\n✓ Macro F1 = {f1_macro:.4f}")

if f1_macro >= 0.65:
    print("  → ОТЛИЧНО! Модель хорошо справляется со всеми классами.")
    print("  → Можно переходить к подбору гиперпараметров (Блок 8).")
elif f1_macro >= 0.55:
    print("  → ХОРОШО! Модель работает, но есть потенциал для улучшения.")
    print("  → Рекомендуется подбор гиперпараметров (Блок 8).")
elif f1_macro >= 0.45:
    print("  → УДОВЛЕТВОРИТЕЛЬНО. Модель учится, но качество низкое.")
    print("  → Сначала проведите анализ ошибок (Блок 7, 10).")
else:
    print("  → ПЛОХО. Модель не справляется с задачей.")
    print("  → Требуется глубокий анализ ошибок и пересмотр признаков.")

# Анализ по классам
print(f"\n📈 Анализ по классам:")

# Извлекаем F1 для каждого класса из отчёта
report_dict = classification_report(y_test, y_pred, output_dict=True, target_names=['Bad', 'Good', 'Neutral'])

for cls in ['Bad', 'Good', 'Neutral']:
    f1_cls = report_dict[cls]['f1-score']
    precision_cls = report_dict[cls]['precision']
    recall_cls = report_dict[cls]['recall']
    
    print(f"\n   Класс '{cls}':")
    print(f"      F1-score  : {f1_cls:.4f}")
    print(f"      Precision : {precision_cls:.4f} (качество предсказаний)")
    print(f"      Recall    : {recall_cls:.4f} (полнота обнаружения)")
    
    if f1_cls < 0.5:
        print(f"      ⚠️  НИЗКОЕ КАЧЕСТВО! Модель плохо распознаёт этот класс.")
    elif f1_cls < 0.65:
        print(f"      ⚡ СРЕДНЕЕ КАЧЕСТВО. Есть потенциал для улучшения.")
    else:
        print(f"      ✓ ХОРОШЕЕ КАЧЕСТВО.")

print("\n" + "=" * 70)
print("✓ Блок 6 завершён!")
print("=" * 70)
print("\nСледующие шаги:")
print("  1. Если Macro F1 >= 0.65 → переход к Блоку 8 (подбор гиперпараметров)")
print("  2. Если Macro F1 < 0.65 → сначала Блок 7 (детальный анализ ошибок)")
print("  3. Для интерпретации модели → Блок 9 (важность признаков)")

БЛОК 6: Базовое обучение логистической регрессии

1. Инициализация модели...
   Модель: LogisticRegression
   Параметры:
      - solver='saga' (поддерживает multiclass)
      - class_weight='balanced' (борьба с дисбалансом)
      - penalty='l2', C=1.0
      - multi_class='multinomial'

2. Обучение модели...
   ✓ Обучение завершено за 17.23 сек.

3. Предсказание на тестовой выборке...
   ✓ Предсказание завершено за 0.01 сек.
   Размер предсказаний: 7316

4. Вычисление метрик качества...

РЕЗУЛЬТАТЫ БАЗОВОГО ОБУЧЕНИЯ

📊 Глобальные метрики:
   Accuracy (точность)     : 0.7966 (79.66%)
   Macro F1-score          : 0.6702 ← ГЛАВНАЯ МЕТРИКА
   Weighted F1-score       : 0.8075
   Micro F1-score          : 0.7966

ДЕТАЛЬНЫЙ ОТЧЁТ ПО КЛАССАМ

              precision    recall  f1-score   support

         Bad     0.6925    0.7989    0.7419       950
        Good     0.9279    0.8573    0.8912      5451
     Neutral     0.3345    0.4328    0.3773       915

    accuracy                         0

#### 📋 План улучшения распознавания Neutral

##### Шаг 1: Анализ ошибок (Error Analysis)

In [19]:
import numpy as np
from sklearn.metrics import confusion_matrix
from collections import Counter

print("=" * 70)
print("АНАЛИЗ ОШИБОК ДЛЯ КЛАССА NEUTRAL")
print("=" * 70)

# ==========================================================================
# 1. Матрица ошибок
# ==========================================================================
cm = confusion_matrix(y_test, y_pred, labels=['Bad', 'Good', 'Neutral'])

print("\nМатрица ошибок (строки = истина, столбцы = предсказание):")
print("              Pred: Bad  Good  Neutral")
print(f"True: Bad     {cm[0][0]:4d}  {cm[0][1]:4d}  {cm[0][2]:4d}")
print(f"      Good    {cm[1][0]:4d}  {cm[1][1]:4d}  {cm[1][2]:4d}")
print(f"      Neutral {cm[2][0]:4d}  {cm[2][1]:4d}  {cm[2][2]:4d}")

# ==========================================================================
# 2. Анализ ошибок Neutral
# ==========================================================================
print("\n" + "=" * 70)
print("ДЕТАЛЬНЫЙ РАЗБОР ОШИБОК NEUTRAL")
print("=" * 70)

# Находим все примеры, где истина = Neutral
neutral_indices = np.where(y_test == 'Neutral')[0]
print(f"\nВсего примеров Neutral в тесте: {len(neutral_indices)}")

# Считаем, как модель их классифицировала
neutral_preds = y_pred[neutral_indices]
neutral_pred_dist = Counter(neutral_preds)

print(f"\nКак модель предсказала Neutral примеры:")
for cls, count in sorted(neutral_pred_dist.items()):
    pct = count / len(neutral_indices) * 100
    print(f"   → {cls:10}: {count:3} ({pct:5.2f}%)")

# ==========================================================================
# 3. Примеры ошибок (ИСПРАВЛЕНО для sparse matrix)
# ==========================================================================
print("\n" + "=" * 70)
print("ПРИМЕРЫ ОШИБОК (Neutral предсказан как другой класс)")
print("=" * 70)

# ИСПРАВЛЕНО: преобразуем X_test в CSR для корректной индексации
X_test_csr = X_test.tocsr()

# Индексы, где истина = Neutral, но предсказание != Neutral
neutral_errors = [i for i in neutral_indices if y_pred[i] != 'Neutral']

print(f"\nВсего ошибок Neutral: {len(neutral_errors)}")

# Показываем 5 примеров
for idx in neutral_errors[:5]:
    print(f"\n{'─' * 70}")
    print(f"ИСТИНА: Neutral | ПРЕДСКАЗАНИЕ: {y_pred[idx]}")
    print(f"{'─' * 70}")
    
    # Метаданные из X_meta_test
    print(f"Длина текста: {X_meta_test_orig[idx, 0]:.0f} символов")
    print(f"Есть заголовок: {'Да' if X_meta_test_scaled[idx, 1] > 0 else 'Нет'}")
    print(f"Есть цифры: {'Да' if X_meta_test_scaled[idx, 2] > 0 else 'Нет'}")
    
    # ИСПРАВЛЕНО: используем CSR матрицу для индексации
    row = X_test_csr[idx]
    probs = model.predict_proba(row)[0]
    
    print(f"\nВероятности модели:")
    for cls, prob in zip(model.classes_, probs):
        bar = '█' * int(prob * 30)
        print(f"   {cls:10}: {prob:.4f} {bar}")

# ==========================================================================
# 4. Примеры успешных предсказаний Neutral (для сравнения)
# ==========================================================================
print("\n" + "=" * 70)
print("ПРИМЕРЫ УСПЕШНЫХ ПРЕДСКАЗАНИЙ NEUTRAL")
print("=" * 70)

neutral_correct = [i for i in neutral_indices if y_pred[i] == 'Neutral']
print(f"\nВсего успешных предсказаний Neutral: {len(neutral_correct)}")

for idx in neutral_correct[:3]:
    print(f"\n{'─' * 70}")
    print(f"ИСТИНА: Neutral | ПРЕДСКАЗАНИЕ: {y_pred[idx]} ✓")
    print(f"{'─' * 70}")
    
    print(f"Длина текста: {X_meta_test_orig[idx, 0]:.0f} символов")
    print(f"Есть заголовок: {'Да' if X_meta_test_scaled[idx, 1] > 0 else 'Нет'}")
    print(f"Есть цифры: {'Да' if X_meta_test_scaled[idx, 2] > 0 else 'Нет'}")
    
    row = X_test_csr[idx]
    probs = model.predict_proba(row)[0]
    
    print(f"\nВероятности модели:")
    for cls, prob in zip(model.classes_, probs):
        bar = '█' * int(prob * 30)
        print(f"   {cls:10}: {prob:.4f} {bar}")

# ==========================================================================
# 5. Анализ текстов Neutral из dataset (для понимания природы класса)
# ==========================================================================
print("\n" + "=" * 70)
print("ПРИМЕРЫ ТЕКСТОВ КЛАССА NEUTRAL ИЗ ДАТАСЕТА")
print("=" * 70)

# Находим примеры Neutral в исходном dataset
neutral_in_dataset = dataset.filter(lambda x: x['grade3'] == 'Neutral')
print(f"\nВсего примеров Neutral в датасете: {len(neutral_in_dataset)}")

# Показываем 5 случайных примеров
import random
random.seed(42)
sample_indices = random.sample(range(len(neutral_in_dataset)), min(5, len(neutral_in_dataset)))

for i in sample_indices:
    sample = neutral_in_dataset[i]
    print(f"\n{'─' * 70}")
    print(f"Фильм: {sample['movie_name']}")
    print(f"Заголовок: {sample['title'] if sample['title'] else 'Без заголовка'}")
    print(f"Длина: {len(sample['content'])} символов")
    print(f"Текст (первые 300 символов):")
    print(f"  {sample['content'][:300]}...")

АНАЛИЗ ОШИБОК ДЛЯ КЛАССА NEUTRAL

Матрица ошибок (строки = истина, столбцы = предсказание):
              Pred: Bad  Good  Neutral
True: Bad      759    45   146
      Good     136  4673   642
      Neutral  201   318   396

ДЕТАЛЬНЫЙ РАЗБОР ОШИБОК NEUTRAL

Всего примеров Neutral в тесте: 915

Как модель предсказала Neutral примеры:
   → Bad       : 201 (21.97%)
   → Good      : 318 (34.75%)
   → Neutral   : 396 (43.28%)

ПРИМЕРЫ ОШИБОК (Neutral предсказан как другой класс)

Всего ошибок Neutral: 519

──────────────────────────────────────────────────────────────────────
ИСТИНА: Neutral | ПРЕДСКАЗАНИЕ: Bad
──────────────────────────────────────────────────────────────────────
Длина текста: 3028 символов
Есть заголовок: Да
Есть цифры: Да

Вероятности модели:
   Bad       : 0.6234 ██████████████████
   Good      : 0.1171 ███
   Neutral   : 0.2596 ███████

──────────────────────────────────────────────────────────────────────
ИСТИНА: Neutral | ПРЕДСКАЗАНИЕ: Good
──────────────────────────

Filter:   0%|          | 0/36579 [00:00<?, ? examples/s]

Filter: 100%|██████████| 36579/36579 [00:00<00:00, 84318.75 examples/s]


Всего примеров Neutral в датасете: 4575

──────────────────────────────────────────────────────────────────────
Фильм: Иллюзионист (2006)
Заголовок: Без заголовка
Длина: 561 символов
Текст (первые 300 символов):
  
В этом фильме ясно улавливается что-то общее с «Ромео и Джульетта» и «Мулен Руж», но тут же неосязаемо отличается от них. Особо хочу отметить иллюзии и фокусы, которые показывал Изенхайм. 

Сначала я подумал, что это обычные фокусы, но ходу просмотра стал думать: Иллюзионист и правда «продал душу д...

──────────────────────────────────────────────────────────────────────
Фильм: В диких условиях (2007)
Заголовок: Свобода и рабство!
Длина: 1784 символов
Текст (первые 300 символов):
  
Понравилось: 

- красивые пейзажи и природа, на фоне которой главный герой осуществлял свое путешествие

- философское завершение истории, которая показала, что: убежать, закрыться в себе, не прощать это не решение проблем, с которыми в той или иной степени сталкиваемся все мы на протяжении жиз

##### Шаг 2: Пороговая настройка (Threshold Tuning)

In [20]:
print("\n" + "=" * 70)
print("ПОРОГОВАЯ НАСТРОЙКА (THRESHOLD TUNING)")
print("=" * 70)

# Получаем вероятности для тестовой выборки
y_probs = model.predict_proba(X_test)

# Создаем словарь классов
class_to_idx = {cls: idx for idx, cls in enumerate(model.classes_)}
neutral_idx = class_to_idx['Neutral']

print(f"\nИндекс класса Neutral: {neutral_idx}")
print(f"Классы модели: {model.classes_}")

# Тестируем разные пороги для Neutral
thresholds = [0.2, 0.25, 0.3, 0.33, 0.35, 0.4, 0.45, 0.5]

print(f"\n{'Порог':<10} {'Accuracy':<12} {'Macro F1':<12} {'Neutral F1':<12}")
print("─" * 50)

best_threshold = 0.5
best_f1_macro = 0

for threshold in thresholds:
    y_pred_custom = []
    
    for probs in y_probs:
        # Если вероятность Neutral выше порога — предсказываем Neutral
        if probs[neutral_idx] >= threshold:
            y_pred_custom.append('Neutral')
        else:
            # Иначе выбираем между Bad и Good
            other_probs = [probs[i] for i in range(len(probs)) if i != neutral_idx]
            other_classes = [cls for cls in model.classes_ if cls != 'Neutral']
            max_idx = other_probs.index(max(other_probs))
            y_pred_custom.append(other_classes[max_idx])
    
    y_pred_custom = np.array(y_pred_custom)
    
    # Считаем метрики
    acc = accuracy_score(y_test, y_pred_custom)
    f1_macro = f1_score(y_test, y_pred_custom, average='macro')
    
    # ИСПРАВЛЕНО: создаем бинарную задачу для Neutral vs Остальное
    y_test_binary = np.array([1 if y == 'Neutral' else 0 for y in y_test])
    y_pred_binary = np.array([1 if y == 'Neutral' else 0 for y in y_pred_custom])
    
    f1_neutral = f1_score(y_test_binary, y_pred_binary, average='binary')
    
    print(f"{threshold:<10.2f} {acc:<12.4f} {f1_macro:<12.4f} {f1_neutral:<12.4f}")
    
    if f1_macro > best_f1_macro:
        best_f1_macro = f1_macro
        best_threshold = threshold

print(f"\n✓ Лучший порог: {best_threshold:.2f} (Macro F1 = {best_f1_macro:.4f})")

# ==========================================================================
# Применение лучшего порога
# ==========================================================================
print("\n" + "=" * 70)
print(f"ПРИМЕНЕНИЕ ЛУЧШЕГО ПОРОГА: {best_threshold:.2f}")
print("=" * 70)

y_pred_final = []

for probs in y_probs:
    if probs[neutral_idx] >= best_threshold:
        y_pred_final.append('Neutral')
    else:
        other_probs = [probs[i] for i in range(len(probs)) if i != neutral_idx]
        other_classes = [cls for cls in model.classes_ if cls != 'Neutral']
        max_idx = other_probs.index(max(other_probs))
        y_pred_final.append(other_classes[max_idx])

y_pred_final = np.array(y_pred_final)

# Финальные метрики
acc_final = accuracy_score(y_test, y_pred_final)
f1_macro_final = f1_score(y_test, y_pred_final, average='macro')

y_test_binary = np.array([1 if y == 'Neutral' else 0 for y in y_test])
y_pred_binary = np.array([1 if y == 'Neutral' else 0 for y in y_pred_final])
f1_neutral_final = f1_score(y_test_binary, y_pred_binary, average='binary')

print(f"\nФинальные метрики после пороговой настройки:")
print(f"   Accuracy   : {acc_final:.4f}")
print(f"   Macro F1   : {f1_macro_final:.4f}")
print(f"   Neutral F1 : {f1_neutral_final:.4f}")

# Сравнение с базовой моделью
print(f"\nСравнение с базовой моделью (Блок 6):")
print(f"   Macro F1 было: {f1_macro:.4f}")
print(f"   Macro F1 стало: {f1_macro_final:.4f}")
print(f"   Улучшение: {f1_macro_final - f1_macro:+.4f}")

# Детальный отчёт
print("\n" + "=" * 70)
print("ДЕТАЛЬНЫЙ ОТЧЁТ ПОСЛЕ ПОРОГОВОЙ НАСТРОЙКИ")
print("=" * 70)

report_final = classification_report(
    y_test, 
    y_pred_final, 
    target_names=['Bad', 'Good', 'Neutral'],
    digits=4
)

print(report_final)


ПОРОГОВАЯ НАСТРОЙКА (THRESHOLD TUNING)

Индекс класса Neutral: 2
Классы модели: ['Bad' 'Good' 'Neutral']

Порог      Accuracy     Macro F1     Neutral F1  
──────────────────────────────────────────────────
0.20       0.5935       0.5564       0.3426      
0.25       0.6633       0.6047       0.3653      
0.30       0.7156       0.6365       0.3847      
0.33       0.7370       0.6466       0.3889      
0.35       0.7478       0.6503       0.3846      
0.40       0.7757       0.6623       0.3769      
0.45       0.7987       0.6675       0.3686      
0.50       0.8115       0.6640       0.3486      

✓ Лучший порог: 0.45 (Macro F1 = 0.6675)

ПРИМЕНЕНИЕ ЛУЧШЕГО ПОРОГА: 0.45

Финальные метрики после пороговой настройки:
   Accuracy   : 0.7987
   Macro F1   : 0.6675
   Neutral F1 : 0.3686

Сравнение с базовой моделью (Блок 6):
   Macro F1 было: 0.6640
   Macro F1 стало: 0.6675
   Улучшение: +0.0035

ДЕТАЛЬНЫЙ ОТЧЁТ ПОСЛЕ ПОРОГОВОЙ НАСТРОЙКИ
              precision    recall  f1-score   s

##### Шаг 4: Добавление признаков "Амбивалентности" (Смешанных чувств)

In [21]:
import re
import numpy as np

print("=" * 70)
print("ШАГ 4: ПРОДВИНУТЫЙ FEATURE ENGINEERING ДЛЯ NEUTRAL")
print("=" * 70)

# Словари маркеров (упрощенные, но эффективные для русского языка)
POSITIVE_MARKERS = {'хорош', 'отличн', 'классн', 'прекрасн', 'замечательн', 'интересн', 'нрави', 'совет'}
NEGATIVE_MARKERS = {'плох', 'ужас', 'скучн', 'разочарова', 'отвратит', 'глуп', 'бред', 'потеря'}
CONTRAST_MARKERS = {'но', 'однако', 'хотя', 'зато', 'тем не менее', 'правда', 'впрочем'}

def extract_ambivalence_features(texts):
    """Извлекает признаки, указывающие на смешанные чувства (нейтральность)"""
    features = {
        'contrast_count': [],       # Количество противопоставлений
        'positive_words_count': [], # Количество позитивных корней
        'negative_words_count': [], # Количество негативных корней
        'is_mixed_sentiment': [],   # 1, если есть И позитив, И негатив (главный маркер Neutral!)
        'question_marks': []        # Количество вопросов (часто бывают в нейтральных/сомневающихся отзывах)
    }
    
    for text in texts:
        text_lower = text.lower()
        words = set(re.findall(r'\b[а-яё]+\b', text_lower)) # Только слова
        
        # 1. Считаем противопоставления
        contrast_count = sum(1 for marker in CONTRAST_MARKERS if marker in text_lower)
        features['contrast_count'].append(contrast_count)
        
        # 2. Считаем позитивные и негативные маркеры (через корни)
        pos_count = sum(1 for w in words for marker in POSITIVE_MARKERS if w.startswith(marker))
        neg_count = sum(1 for w in words for marker in NEGATIVE_MARKERS if w.startswith(marker))
        
        features['positive_words_count'].append(pos_count)
        features['negative_words_count'].append(neg_count)
        
        # 3. Главный признак: смешанный сентимент (есть и то, и другое)
        is_mixed = 1 if (pos_count > 0 and neg_count > 0) else 0
        features['is_mixed_sentiment'].append(is_mixed)
        
        # 4. Знаки вопроса (сомнение)
        features['question_marks'].append(text.count('?'))
        
    return features

# Применяем ко всему датасету
print("Извлечение новых признаков...")
new_features = extract_ambivalence_features(dataset['content_clean'])

# Добавляем новые колонки в датасет
for col_name, values in new_features.items():
    dataset = dataset.add_column(col_name, values)

print("✓ Новые признаки добавлены в датасет.")
print(f"  Пример: is_mixed_sentiment = {dataset['is_mixed_sentiment'][:5]}")

ШАГ 4: ПРОДВИНУТЫЙ FEATURE ENGINEERING ДЛЯ NEUTRAL
Извлечение новых признаков...
✓ Новые признаки добавлены в датасет.
  Пример: is_mixed_sentiment = [1, 1, 0, 0, 1]


##### Шаг 5: Переобучение модели с новыми признаками

In [22]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, precision_recall_fscore_support
from sklearn.feature_extraction.text import TfidfVectorizer

print("\n" + "=" * 70)
print("ПЕРЕОБУЧЕНИЕ МОДЕЛИ С НОВЫМИ ПРИЗНАКАМИ")
print("=" * 70)

# 1. Собираем расширенный набор мета-признаков
X_meta_extended = np.column_stack([
    dataset['content_len'],
    dataset['has_title'],
    dataset['has_numbers'],
    dataset['contrast_count'],
    dataset['positive_words_count'],
    dataset['negative_words_count'],
    dataset['is_mixed_sentiment'],      # <-- Самый важный новый признак
    dataset['question_marks']
])

y = np.array(dataset['grade3'])

# 2. Разбиение (стратифицированное)
X_text = np.array(dataset['content_clean'])
X_text_train, X_text_test, X_meta_train_ext, X_meta_test_ext, y_train, y_test = train_test_split(
    X_text, X_meta_extended, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. Векторизация текста
tfidf_new = TfidfVectorizer(ngram_range=(1, 2), max_features=50000, min_df=3, max_df=0.9, sublinear_tf=True)
X_text_train_tfidf = tfidf_new.fit_transform(X_text_train)
X_text_test_tfidf = tfidf_new.transform(X_text_test)

# 4. Масштабирование мета-признаков
scaler_ext = StandardScaler()
X_meta_train_scaled = scaler_ext.fit_transform(X_meta_train_ext)
X_meta_test_scaled = scaler_ext.transform(X_meta_test_ext)

# 5. Объединение
X_train_ext = hstack([X_text_train_tfidf, X_meta_train_scaled])
X_test_ext = hstack([X_text_test_tfidf, X_meta_test_scaled])

# 6. Обучение модели
print("Обучение модели с расширенными признаками...")
model_ext = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    solver='saga',
    penalty='l2',
    C=1.0,
    random_state=42
)
model_ext.fit(X_train_ext, y_train)

# 7. Предсказание и оценка
y_pred_ext = model_ext.predict(X_test_ext)

print("\nРЕЗУЛЬТАТЫ ПОСЛЕ ДОБАВЛЕНИЯ ПРИЗНАКОВ АМБИВАЛЕНТНОСТИ:")
report = classification_report(y_test, y_pred_ext, target_names=['Bad', 'Good', 'Neutral'], digits=4)
print(report)

# ИСПРАВЛЕНО: Правильный способ получить F1 для конкретного класса в multiclass
# Используем average=None и извлекаем результат для 'Neutral'
prec_all, rec_all, f1_all, _ = precision_recall_fscore_support(
    y_test, y_pred_ext, labels=['Bad', 'Good', 'Neutral'], average=None
)

# Находим индекс класса 'Neutral'
classes = ['Bad', 'Good', 'Neutral']
neutral_idx = classes.index('Neutral')

f1_neutral_ext = f1_all[neutral_idx]
prec_neutral_ext = prec_all[neutral_idx]
rec_neutral_ext = rec_all[neutral_idx]

print("\n" + "=" * 70)
print("🎯 РЕЗУЛЬТАТЫ ДЛЯ КЛАССА NEUTRAL")
print("=" * 70)
print(f"   Precision: {prec_neutral_ext:.4f}")
print(f"   Recall   : {rec_neutral_ext:.4f}")
print(f"   F1-score : {f1_neutral_ext:.4f}")

# Также считаем Macro F1
f1_macro_ext = f1_score(y_test, y_pred_ext, average='macro')
print(f"\n   Macro F1 (все классы): {f1_macro_ext:.4f}")

# Сравнение с предыдущими результатами
print("\n" + "=" * 70)
print("СРАВНЕНИЕ РЕЗУЛЬТАТОВ")
print("=" * 70)
print(f"   Базовая модель (TF-IDF только):")
print(f"      Neutral F1 ≈ 0.3686")
print(f"      Macro F1 ≈ {f1_macro:.4f}")
print(f"\n   Модель с признаками амбивалентности:")
print(f"      Neutral F1 = {f1_neutral_ext:.4f}")
print(f"      Macro F1 = {f1_macro_ext:.4f}")

if f1_neutral_ext > 0.3686:
    improvement = f1_neutral_ext - 0.3686
    print(f"\n   ✓ УЛУЧШЕНИЕ Neutral F1: +{improvement:.4f}")
else:
    print(f"\n   ✗ Улучшения нет или результат хуже")

print("\n" + "=" * 70)
print("ИНТЕРПРЕТАЦИЯ")
print("=" * 70)

if f1_neutral_ext >= 0.45:
    print("✓ ОТЛИЧНО! Признаки амбивалентности помогли.")
    print("  Можно переходить к подбору гиперпараметров (Блок 8).")
elif f1_neutral_ext >= 0.40:
    print("⚡ ХОРОШО! Есть улучшение, но можно улучшить дальше.")
    print("  Рекомендуется попробовать:")
    print("    1. Подбор гиперпараметров (Блок 8)")
    print("    2. Пороговую настройку (Шаг 2) с новой моделью")
else:
    print("⚠️  Улучшение минимально или отсутствует.")
    print("  Линейная модель + TF-IDF достигла своего потолка.")
    print("  Рекомендуется сменить архитектуру:")
    print("    1. CatBoost/LightGBM (лучше работает с мета-признаками)")
    print("    2. Нейросеть (ruBERT-tiny2) для понимания контекста")


ПЕРЕОБУЧЕНИЕ МОДЕЛИ С НОВЫМИ ПРИЗНАКАМИ
Обучение модели с расширенными признаками...

РЕЗУЛЬТАТЫ ПОСЛЕ ДОБАВЛЕНИЯ ПРИЗНАКОВ АМБИВАЛЕНТНОСТИ:
              precision    recall  f1-score   support

         Bad     0.6918    0.7916    0.7383       950
        Good     0.9283    0.8521    0.8886      5451
     Neutral     0.3339    0.4470    0.3822       915

    accuracy                         0.7936      7316
   macro avg     0.6513    0.6969    0.6697      7316
weighted avg     0.8232    0.7936    0.8057      7316


🎯 РЕЗУЛЬТАТЫ ДЛЯ КЛАССА NEUTRAL
   Precision: 0.3339
   Recall   : 0.4470
   F1-score : 0.3822

   Macro F1 (все классы): 0.6697

СРАВНЕНИЕ РЕЗУЛЬТАТОВ
   Базовая модель (TF-IDF только):
      Neutral F1 ≈ 0.3686
      Macro F1 ≈ 0.6640

   Модель с признаками амбивалентности:
      Neutral F1 = 0.3822
      Macro F1 = 0.6697

   ✓ УЛУЧШЕНИЕ Neutral F1: +0.0136

ИНТЕРПРЕТАЦИЯ
⚠️  Улучшение минимально или отсутствует.
  Линейная модель + TF-IDF достигла своего потолка.
  Р

### LLM

#### Блок A: Абстрактный интерфейс модели

In [23]:
%pip install transformers[torch]

Note: you may need to restart the kernel to use updated packages.


In [24]:
from abc import ABC, abstractmethod
from typing import List, Union, Dict, Any
import numpy as np

class TextClassifier(ABC):
    """
    Абстрактный интерфейс для всех моделей классификации текста.
    Позволяет менять модель без изменения пайплайна.
    """
    
    @abstractmethod
    def train(self, texts: List[str], labels: List[str], **kwargs) -> None:
        """Обучение модели на текстах и метках"""
        pass
    
    @abstractmethod
    def predict(self, texts: List[str]) -> np.ndarray:
        """Предсказание классов"""
        pass
    
    @abstractmethod
    def predict_proba(self, texts: List[str]) -> np.ndarray:
        """Предсказание вероятностей классов"""
        pass
    
    @abstractmethod
    def save(self, path: str) -> None:
        """Сохранение модели"""
        pass
    
    @classmethod
    @abstractmethod
    def load(cls, path: str) -> 'TextClassifier':
        """Загрузка модели"""
        pass
    
    @abstractmethod
    def get_model_info(self) -> Dict[str, Any]:
        """Информация о модели (для логирования)"""
        pass


print("✓ Блок A: Абстрактный интерфейс TextClassifier создан")
print("  Методы: train, predict, predict_proba, save, load, get_model_info")

✓ Блок A: Абстрактный интерфейс TextClassifier создан
  Методы: train, predict, predict_proba, save, load, get_model_info


#### Блок B: Реализация для rubert-tiny2

In [37]:
import os
import json
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# ==============================================================================
# КАСТОМНЫЙ TRAINER ДЛЯ ВЗВЕШЕННОЙ ФУНКЦИИ ПОТЕРЬ
# ==============================================================================
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None:
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32)
        else:
            self.class_weights = None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        if self.class_weights is not None:
            # Переносим веса на то же устройство, что и логиты (CPU или GPU)
            weights = self.class_weights.to(logits.device)
            loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fct = torch.nn.CrossEntropyLoss()
            
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ==============================================================================
# ОСНОВНОЙ КЛАСС
# ==============================================================================
class BertClassifier:
    def __init__(
        self,
        model_name: str = "ai-forever/ruBERT-base", # <-- МЕНЯЕМ НА BASE ПО УМОЛЧАНИЮ
        num_labels: int = 3,
        max_length: int = 512,
        device: str = None,
    ):
        self.model_name = model_name
        self.num_labels = num_labels
        self.max_length = max_length

        self.device = torch.device(
            device if device is not None else ("cuda" if torch.cuda.is_available() else "cpu")
        )
        print(f"Модель: {model_name} | Устройство: {self.device}")

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=num_labels
        )
        self.model.to(self.device)
        self.label2id = None
        self.id2label = None

    def _build_mapping(self, labels):
        if self.label2id is None:
            unique = sorted(set(labels))
            self.label2id = {label: i for i, label in enumerate(unique)}
            self.id2label = {i: label for label, i in self.label2id.items()}
            self.model.config.label2id = self.label2id
            self.model.config.id2label = self.id2label
            print(f"Мэппинг классов: {self.label2id}")

    def _prepare_dataset(self, texts, labels=None):
        if labels is not None:
            self._build_mapping(labels)

        data = {"text": list(texts)}
        if labels is not None:
            data["label"] = [self.label2id[label] for label in labels]

        ds = Dataset.from_dict(data)
        def tokenize(examples):
            return self.tokenizer(
                examples["text"], truncation=True, padding=False, max_length=self.max_length
            )
        return ds.map(tokenize, batched=True, remove_columns=["text"])

    def _compute_metrics(self, eval_pred):
        logits, labels = eval_pred
        if hasattr(logits, "detach"): logits = logits.detach().cpu().numpy()
        if hasattr(labels, "cpu"): labels = labels.cpu().numpy()

        preds = np.argmax(logits, axis=-1)
        f1_per_class = f1_score(labels, preds, labels=list(range(self.num_labels)), average=None, zero_division=0)

        metrics = {
            "f1_macro": float(f1_score(labels, preds, average="macro", zero_division=0)),
            "accuracy": float(accuracy_score(labels, preds)),
        }
        metrics.update({f"f1_{self.id2label[i]}": float(f1_per_class[i]) for i in range(self.num_labels)})
        return metrics

    def train(self, texts, labels, val_texts=None, val_labels=None, 
              learning_rate=2e-5, num_epochs=3, batch_size=16, output_dir="./bert_output", neutral_boost=1.0):
        
        train_dataset = self._prepare_dataset(texts, labels)
        eval_dataset = self._prepare_dataset(val_texts, val_labels) if val_texts else None

        # --- РАСЧЕТ ВЕСОВ КЛАССОВ С УСИЛЕНИЕМ NEUTRAL ---
        class_weights = compute_class_weight(
            class_weight='balanced',
            classes=np.array(list(self.label2id.values())),
            y=np.array([self.label2id[l] for l in labels])
        )

        # Усиливаем вес Neutral, если задан neutral_boost
        if neutral_boost > 1.0:
            neutral_idx = self.label2id['Neutral']
            class_weights[neutral_idx] *= neutral_boost
            print(f"⚡ Усиление веса Neutral: x{neutral_boost}")

        print(f"Веса классов для Loss Function: {dict(zip(self.id2label.values(), np.round(class_weights, 2)))}")

        training_args = TrainingArguments(
            output_dir=output_dir,
            learning_rate=learning_rate,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            num_train_epochs=num_epochs,
            weight_decay=0.01,
            eval_strategy="epoch" if eval_dataset else "no",
            save_strategy="epoch",
            load_best_model_at_end=eval_dataset is not None,
            metric_for_best_model="f1_macro",
            greater_is_better=True,
            logging_steps=10,
            save_total_limit=2,
            report_to="none",
            fp16=(self.device.type == "cuda"),
            remove_unused_columns=True,
        )

        # Используем наш кастомный Trainer с весами
        trainer = WeightedTrainer(
            class_weights=class_weights, # <-- Передаем веса
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            data_collator=DataCollatorWithPadding(tokenizer=self.tokenizer),
            compute_metrics=self._compute_metrics,
        )

        print("\nЗапуск обучения...")
        train_result = trainer.train()
        
        # Сохранение
        os.makedirs(output_dir, exist_ok=True)
        trainer.save_model(output_dir)
        self.tokenizer.save_pretrained(output_dir)
        self.model = trainer.model
        
        with open(os.path.join(output_dir, "label_mapping.json"), "w", encoding="utf-8") as f:
            json.dump({"label2id": self.label2id, "id2label": {str(k): v for k, v in self.id2label.items()}}, f)

        return dict(train_result.metrics)

    def predict_proba(self, texts, batch_size=32):
        self.model.eval()
        all_probs = []
        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                batch = texts[i:i + batch_size]
                inputs = self.tokenizer(batch, truncation=True, padding=True, max_length=self.max_length, return_tensors="pt").to(self.device)
                logits = self.model(**inputs).logits
                all_probs.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
        return np.concatenate(all_probs, axis=0)

    def predict(self, texts):
        probs = self.predict_proba(texts)
        return np.array([self.id2label[i] for i in probs.argmax(axis=-1)])

print("✓ Обновленный BertClassifier с WeightedTrainer загружен")

✓ Обновленный BertClassifier с WeightedTrainer загружен


#### Блок C: Подготовка данных для BERT

In [26]:
from sklearn.model_selection import train_test_split

print("\n" + "=" * 70)
print("БЛОК C: ПОДГОТОВКА ДАННЫХ ДЛЯ BERT")
print("=" * 70)

# Извлекаем данные из dataset
X_text = dataset['content_clean']  # Используем очищенные тексты
y = dataset['grade3']

print(f"\nИсходные данные:")
print(f"  Количество примеров: {len(X_text)}")
print(f"  Классы: {sorted(set(y))}")

# Стратифицированное разбиение
X_train, X_test, y_train, y_test = train_test_split(
    X_text, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Дополнительное разбиение train на train/val (для early stopping)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1,  # 10% от train = 8% от всего
    random_state=42,
    stratify=y_train
)

print(f"\nРазбиение данных:")
print(f"  Train: {len(X_train)} примеров")
print(f"  Validation: {len(X_val)} примеров")
print(f"  Test: {len(X_test)} примеров")

# Пример текста
print(f"\nПример текста для BERT:")
print(f"  {X_train[0][:200]}...")

print("\n✓ Данные готовы для обучения BERT")


БЛОК C: ПОДГОТОВКА ДАННЫХ ДЛЯ BERT

Исходные данные:
  Количество примеров: 36579
  Классы: ['Bad', 'Good', 'Neutral']

Разбиение данных:
  Train: 26336 примеров
  Validation: 2927 примеров
  Test: 7316 примеров

Пример текста для BERT:
  смешно грустно наверно именно можно описать состояние просмотр фильм жалость именно такой лёгкий щемить грусть невероятно трогательный герой улыбаться хотеться делать вместе естественно плакать потому...

✓ Данные готовы для обучения BERT


#### Блок D: Обучение модели

In [38]:
print("\n" + "=" * 70)
print("ОБУЧЕНИЕ ruBERT-base С ВЗВЕШЕННОЙ ФУНКЦИЕЙ ПОТЕРЬ")
print("=" * 70)

# 1. Инициализируем более мощную модель
classifier = BertClassifier(
    model_name="ai-forever/ruBERT-base", # <-- СМЕНА МОДЕЛИ (требует ~2-4 GB VRAM)
    num_labels=3,
    max_length=256,
    device='cuda' # Уменьшаем до 256 для экономии памяти и ускорения (медиана текста ~1500 символов)
)

# 2. Запускаем обучение
metrics = classifier.train(
    texts=X_train,
    labels=y_train,
    val_texts=X_val,
    val_labels=y_val,
    learning_rate=2e-5,
    num_epochs=3,
    batch_size=16,
    output_dir="./rubert_base_weighted_v2",
    neutral_boost=1.5
)

print("\nИтоговые метрики обучения:")
for k, v in metrics.items():
    if "f1" in k or "loss" in k:
        print(f"  {k}: {v:.4f}")


ОБУЧЕНИЕ ruBERT-base С ВЗВЕШЕННОЙ ФУНКЦИЕЙ ПОТЕРЬ
Модель: ai-forever/ruBERT-base | Устройство: cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 39717.65it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: ai-forever/ruBERT-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archite

Мэппинг классов: {'Bad': 0, 'Good': 1, 'Neutral': 2}


Map: 100%|██████████| 2927/2927 [00:01<00:00, 1722.89 examples/s]


⚡ Усиление веса Neutral: x1.5
Веса классов для Loss Function: {'Bad': np.float64(2.57), 'Good': np.float64(0.45), 'Neutral': np.float64(4.0)}

Запуск обучения...


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy,F1 Bad,F1 Good,F1 Neutral
1,0.790294,0.756505,0.587605,0.631705,0.712395,0.745592,0.304828
2,0.677434,0.772509,0.624192,0.722241,0.695438,0.835411,0.341727
3,0.549805,0.861968,0.629245,0.731466,0.692407,0.846406,0.348921


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.57it/s]


Итоговые метрики обучения:
  train_loss: 0.7001
